####Connection

In [0]:
%sql
--listando arquivos no volume bronze do databricks

list '/Volumes/vg_sales/01_bronze/volume_bronze'

In [0]:
import json

# Caminho onde você salvou o arquivo no Volume
caminho_config = "/Volumes/vg_sales/01_bronze/volume_bronze_confgs/keys.json"

# Lendo o arquivo de forma segura
with open(caminho_config, "r") as f:
    config = json.load(f)

# Atribuindo às variáveis
ACCESS_KEY  = config["ACCESS_KEY"]
SECRET_KEY  = config["SECRET_KEY"]
BUCKET_NAME = config["BUCKET_NAME"]
ENDPOINT    = config["ENDPOINT"]
LOCAL_PATH  = config["LOCAL_PATH"]
PADRAO      = config["PADRAO"]

print(f"Configurações carregadas com sucesso para o bucket: {BUCKET_NAME}")

In [0]:
#Conexao com ambiente supabase, comparaçao de arquivos entre supabase e volume databricks e download apenas de novos arquivos.

import os
import boto3


s3 = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    endpoint_url=ENDPOINT
)


def sincronizar_arquivos():
    print(f"--- Iniciando Comparação ---")
    
    # 1. Obter o que já existe no Databricks (Local)
    # Se a pasta não existir, criamos e retornamos um set vazio
    if not os.path.exists(LOCAL_PATH):
        os.makedirs(LOCAL_PATH, exist_ok=True)
        arquivos_locais = set()
    else:
        arquivos_locais = set(os.listdir(LOCAL_PATH))
    
    print(f"Arquivos no Databricks: {len(arquivos_locais)}")

    # 2. Listar o que existe no Supabase (Remoto)
    response = s3.list_objects_v2(Bucket=BUCKET_NAME)
    
    if 'Contents' not in response:
        print("O bucket está vazio.")
        return

    novos_arquivos = []

    # 3. Filtrar e Comparar
    for obj in response['Contents']:
        nome_arquivo = obj['Key']
        
        # Só olhamos se tiver o padrão vg_sales_raw
        if PADRAO in nome_arquivo:
            # Se NÃO estiver na lista local, adicionamos à lista de download
            if nome_arquivo not in arquivos_locais:
                novos_arquivos.append(nome_arquivo)

    # 4. Resultado e Ação
    if novos_arquivos:
        print(f"Encontrados {len(novos_arquivos)} novos arquivos para baixar.")
        for arquivo in novos_arquivos:
            print(f"-> Baixando: {arquivo}")
            caminho_destino = os.path.join(LOCAL_PATH, arquivo)
            
            try:
                s3.download_file(BUCKET_NAME, arquivo, caminho_destino)
                print(f"   [OK] {arquivo} salvo com sucesso.")
            except Exception as e:
                print(f"   [ERRO] Falha ao baixar {arquivo}: {e}")
    else:
        print("Tudo atualizado! Nenhum arquivo novo encontrado no Supabase.")

# Executa a função
sincronizar_arquivos()